In [87]:
import pandas as pd
import numpy as np

In [88]:
train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')
pd.set_option('display.max_colwidth', None)
print(train_df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [89]:
train_df.head(n=1)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,"Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.","Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.","Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.",Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present are interconnected and the future is predetermined. Human beings do not have free will.,"Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneously. Humans exist outside of this illusion and are guided by a higher power.",B


ques 1: Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  


In [90]:
answer_counts = train_df["answer"].value_counts()

print(answer_counts)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [91]:
print("Answer 1:")
print(490+324)

Answer 1:
814


ques 2: After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  


In [ ]:
import string 

translator = str.maketrans('', '', string.punctuation)

def clean_text(text):
    text = str(text).lower()
    text = text.translate(translator)
    return text

vocabulary = set()

for text in train_df["prompt"]:
    cleaned = clean_text(text)
    vocabulary.update(cleaned.split())

print("Answer 2:")
print(len(vocabulary))


Answer 2:
859


ques 3: Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  


In [93]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

train_df["prompt"] = train_df["prompt"].apply(clean_text)

tokens = train_df.iloc[0]["prompt"].split()

filtered = [word for word in tokens if word not in ENGLISH_STOP_WORDS]

print("Answer 3")
print(len(filtered))


Answer 3
13


ques 4: Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  


In [94]:
train_df = pd.read_csv('../dataset/train.csv')


combined_texts = []
options = ['A', 'B', 'C', 'D', 'E']

for _, row in train_df.iterrows():

    combined = str(row['prompt'])

    for col in options:
        combined += " " + str(row[col])

    combined_texts.append(combined)

vec = TfidfVectorizer(stop_words='english')
vec.fit(combined_texts)


print("Answer 4")
print(len(vec.vocabulary_))


Answer 4
2762


question 5:  Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  


In [101]:
from sklearn.metrics.pairwise import cosine_similarity

row = train_df.iloc[0]

prompt_vec = vec.transform([row['prompt']])
opt_vec = vec.transform([row['A']])

sim = cosine_similarity(prompt_vec, opt_vec)

print("Answer 5")
print(sim)
print("Rounded to 4 digits:")
print(round(sim[0][0], 4))

Answer 5
[[0.2720243]]
Rounded to 4 digits:
0.272


ques 6: Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   


In [96]:
correct = 0
options = ['A', 'B', 'C', 'D', 'E']
for _, row in train_df.iterrows():

    scores = {}
    prompt_vec = vec.transform([row['prompt']])

    for opt in options:

        opt_vec = vec.transform([row[opt]])

        score = cosine_similarity(prompt_vec, opt_vec)[0][0]
        scores[opt] = score

    best = max(scores, key = scores.get)

    if best == row['answer']:
        correct+=1


percentage = (correct/len(train_df))*100

print("Answer 6")
print(percentage, "%")

    

Answer 6
13.55 %


ques 7: If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  


In [97]:
preds = ["C", "A", "B"]
truth = "C"

if truth in preds:
    rank = preds.index(truth) + 1
    map3 = 1 / rank
else:
    map3 = 0

print("Answer 7")
print(map3)

Answer 7
1.0


ques 8: If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  


In [98]:
preds = ["D", "B", "E"]
truth = "B"

if truth in preds:
    rank = preds.index(truth) + 1
    map3 = 1 / rank
else:
    map3 = 0

print("Answer 8")
print(map3)

Answer 8
0.5


ques 9: The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?



In [99]:
answer_counts = train_df["answer"].value_counts()
ans = answer_counts[:3].keys().to_list()
print(ans)

scores = []

for val in train_df['answer']:
    if val in ans:
        rank = ans.index(val) + 1
        scores.append(1/rank)
    else:
        scores.append(0)

majority_ans = sum(scores)/len(scores)

print("Answer 9")
print(majority_ans)


['B', 'C', 'A']
Answer 9
0.42125


question 10: The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  


In [100]:
scores = []
options = ['A', 'B', 'C', 'D', 'E']

for _, row in train_df.iterrows():
    prompt = row['prompt']
    prompt_vec = vec.transform([prompt])

    option_scores= {}

    for opt in options:
        opt_vec = vec.transform([row[opt]])

        sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        option_scores[opt] = sim

    ranked = sorted(option_scores.items(), key = lambda x: x[1], reverse=True)

    top3 = [x[0] for x in ranked[:3]]

    truth = row['answer']

    if truth in top3:
        rank = top3.index(truth) + 1
        scores.append(1/rank)
    else:
        scores.append(0)

final_map3 = sum(scores) / len(scores)

print("Answer 10")
print(final_map3)


            

Answer 10
0.2961666666666667
